# MAP — Quickstart

**Model Action Protocol** is an open standard for cryptographically-verifiable logs of autonomous AI agent actions. Every tool call an agent makes becomes a hash-chained ledger entry. Reversers are first-class. Cross-language conformance is part of the spec.

This notebook walks through the canonical scenario: an agent places a customer order, a critic flags an outlier, and you roll back to a known-good state. Same flow as `tests/test_sdk_integration.py` and the FastAPI demo at `examples/fastapi_app/`.

Specification: [`spec/SPEC.md`](../../spec/SPEC.md). Design choices: [`DESIGN.md`](../DESIGN.md).

In [ ]:
# Install (run once):
# !pip install "map-protocol[anthropic]"

from typing import Any

from map import (
    Action,
    CriticResult,
    Map,
    rule_critic,
    verify_chain,
    SPEC_VERSION,
    __version__,
)

print(f"MAP Python {__version__} — conforms to spec v{SPEC_VERSION}")

## The problem

An agent places customer orders against a fulfillment system. You need an audit trail: every order should be cryptographically attributable to the agent that placed it, and any single bad call should be reversible without rebuilding state from logs.

The tool itself is whatever you'd write today — a function that talks to your backend. The job MAP does is *wrap* that tool: capture the call, record the verdict from a reviewer, make rollback first-class.

Define the tool and its reverser:

In [ ]:
# A tiny in-memory "orders database" — stand-in for a real fulfillment service.
ORDERS: dict[str, dict[str, Any]] = {}

def place_order(item_id: str, quantity: int) -> dict[str, Any]:
    """Place a customer order."""
    order_id = f"O-{item_id}-{quantity}-{len(ORDERS)}"
    record = {"orderId": order_id, "item_id": item_id, "quantity": quantity, "status": "open"}
    ORDERS[order_id] = record
    return record

def cancel_order(action: Action, output: Any) -> dict[str, Any]:
    """Reverser — flips the order's status to 'cancelled'."""
    order_id = output["orderId"]
    ORDERS[order_id]["status"] = "cancelled"
    return {"orderId": order_id, "cancelled": True}

## Wrap with MAP

Three things happen here:

1. **Critic** — a tiny rule that flags any order over 100 units. In production you'd compose an LLM critic via `map.llm_critic(...)`; for the notebook we keep it deterministic.
2. **Reversible decorator** — registers `cancel_order` as the reverser for `place_order`. If we roll back later, MAP knows what to call.
3. **`Map` instance** — owns the ledger, the reverser registry, and the critic. One per agent.

In [ ]:
def disallow_huge_orders(action: Action, sb: Any, sa: Any) -> CriticResult | None:
    if action.tool == "place_order" and action.input.get("quantity", 0) > 100:
        return CriticResult(
            verdict="FLAGGED",
            reason="quantity over 100 requires human approval",
        )
    return None

m = Map()
m.set_critic(rule_critic([disallow_huge_orders]))
decorated_place_order = m.reversible(reverser=cancel_order)(place_order)

print("Map ready. Tool registered:", m.reversers.known_tools())

## Place a few orders

We call `decorated_place_order(...)` directly to get the output (this is the same shape as if Anthropic's tool-use loop had called it via `wrap_tool_call`). Then we hand the output to `m.execute(...)` to record the call as a ledger entry.

In [ ]:
for item_id, qty in [("SKU-A", 1), ("SKU-B", 5), ("SKU-X", 500)]:
    output = decorated_place_order(item_id=item_id, quantity=qty)
    m.execute(Action(tool="place_order", input={"item_id": item_id, "quantity": qty}, output=output))

for entry in m.get_entries():
    print(
        f"#{entry.sequence} {entry.action.tool}({entry.action.input})"
        f"  →  verdict={entry.critic.verdict}"
    )

## What the ledger captured

Every entry is hash-chained to the previous one. Even the FLAGGED order is in the ledger — the critic's job is to mark, not block. (The agent loop owns the halt-vs-continue decision, not MAP.)

The ledger is verifiable as a self-consistent chain. Tamper with any byte of any entry's hashed fields and `verify_chain` will pinpoint the index.

In [ ]:
chain = [e.model_dump(by_alias=True, exclude_none=True) for e in m.get_entries()]
print("chain valid?", verify_chain(chain))
print("\nstats:", m.get_stats().model_dump(by_alias=True))

## Rollback

Roll back to the first order. MAP walks reversers **newest-first** (saga compensation order) — `SKU-X` is reversed first, then `SKU-B`, then `SKU-A`. World state is updated; the ledger gets a `ROLLBACK` provenance entry.

**Important** ([DESIGN.md §2](../DESIGN.md)): if any reverser raises, MAP propagates the exception and leaves the ledger untouched. World-side effects from already-completed reversers persist — MAP can't un-cancel an order that was just cancelled.

In [ ]:
first_order = m.get_entries()[0]
m.rollback_to(first_order.id)

print("orders after rollback:")
for order_id, record in ORDERS.items():
    print(f"  {order_id}: status={record['status']}")

print("\nstats after rollback:", m.get_stats().model_dump(by_alias=True))
print("\nlast ledger entry:", m.get_entries()[-1].action.tool)

## Cross-language conformance — proof, not handwaving

MAP's claim is that a ledger written in one language verifies byte-identical in another. Both the Python and TypeScript reference implementations conform to the worked example in [SPEC.md §6.4](../../spec/SPEC.md). Compute the canonical bytes of a known entry; the SHA-256 must match the documented hash exactly.

In [ ]:
from map import compute_entry_hash, state_hash, GENESIS_HASH

null_state = state_hash(None)
h = compute_entry_hash(
    sequence=0,
    action={"tool": "ping", "input": {}, "output": "pong"},
    state_before=null_state,
    state_after=null_state,
    parent_hash=GENESIS_HASH,
    critic={"verdict": "PASS", "reason": "ok"},
)

print("computed hash:", h)
expected = "25d29bc25a183ebdb29b70b6a03ed2ad8d31033d1fb6347f656b21d7e9efb650"
print("matches spec?", h == expected)

## Where to go next

- **Spec:** [`spec/SPEC.md`](../../spec/SPEC.md) — wire format, JCS canonicalization, hash rules. ~10 pages, normative.
- **HTTP service:** [`examples/fastapi_app/`](./fastapi_app/) — same scenario as a real backend.
- **SDK integration test:** [`tests/test_sdk_integration.py`](../tests/test_sdk_integration.py) — `wrap_tool_call(...)` for the Anthropic tool-use loop. The live variant in `test_sdk_integration_live.py` runs against the real API.
- **Persistent ledger:** swap `Map()` for `Map(store=SQLiteLedgerStore("ledger.db"))`. For multi-process, use `PostgresLedgerStore` with the `[postgres]` extra.
- **Learning engine:** `LearningEngine().analyze_patterns(m.get_entries())` extracts correction patterns. After repeated CORRECTED entries on the same tool, propose a deterministic rule and run the LLM critic less often.

Found a bug or have a question? File an issue at [DeadpxlStudio/ModelActionProtocol](https://github.com/DeadpxlStudio/ModelActionProtocol).